# Import Data

In [1]:
import pandas as pd
import numpy as np
import pickle
from utils import model as mod
# from utils import results as res

In [2]:
df = pd.read_pickle('../data/03_df_data')
segs = pd.read_pickle('../data/02_df_seg_gen')

with open("../data/02_dic_ref_groups.pkl", "rb") as file:
    ref_groups = pickle.load(file)

In [3]:
df.head()

,loan_type,loan_purpose,preapproval,construction_method,loan_amount,action_taken,state_code,county_code,census_tract,applicant_ethnicity_1,...,loan_term,intro_rate_period,balloon_payment,interest_only_payment,property_value,manufactured_home_secured_property_type,manufactured_home_land_property_interest,reg_uw,denied,reg_price
1,Conventional,Home purchase,Not requested,Site built,1205000,Purchased,WA,53033.0,53033032318.0,Not Hispanic Latino,...,360.0,NaN,No balloon,Not interest only,1505000.0,Not applicable,Not applicable,0,0,0
2,Conventional,Home purchase,Not requested,Site built,925000,Purchased,WA,53011.0,53011040303.0,Info not provided,...,360.0,NaN,No balloon,Not interest only,1035000.0,Not applicable,Not applicable,0,0,0
5,Conventional,Home purchase,Not requested,Site built,905000,Purchased,WA,53011.0,53011040905.0,Not Hispanic Latino,...,360.0,NaN,No balloon,Not interest only,1135000.0,Not applicable,Not applicable,0,0,0
13,Conventional,Home purchase,Not requested,Site built,1505000,Purchased,CA,6065.0,6065045606.0,Not Hispanic Latino,...,360.0,NaN,No balloon,Not interest only,4295000.0,Not applicable,Not applicable,0,0,0
15,Conventional,Refinance,Not requested,Site built,1145000,Purchased,WA,53033.0,53033002700.0,Info not provided,...,360.0,84.0,No balloon,Not interest only,1725000.0,Not applicable,Not applicable,0,0,0


In [4]:
segs

,loan_type,loan_purpose,applicant_sex,applied,event_count,event_rate,mi_rate,mu_rate,mx_rate,crit event,crit count
1,Conventional,Cash out refinance,Female,4610,552,0.119740,4.490,6.698132,9.740,True,True
3,Conventional,Cash out refinance,Male,5713,827,0.144758,2.875,6.798274,10.240,True,True
5,Conventional,Home improvement,Female,1514,255,0.168428,4.875,7.730416,10.240,True,True
7,Conventional,Home improvement,Male,2364,457,0.193316,4.625,7.923380,10.240,True,True
9,Conventional,Home purchase,Female,15263,1029,0.067418,1.000,6.347577,9.365,True,True
11,Conventional,Home purchase,Male,23585,1556,0.065974,2.375,6.362357,9.490,True,True
13,Conventional,Refinance,Female,2964,210,0.070850,4.250,6.192842,8.250,True,True
15,Conventional,Refinance,Male,4790,359,0.074948,4.375,6.164734,8.125,True,True
17,FHA insured,Cash out refinance,Female,582,151,0.259450,4.750,6.181381,7.750,True,True
18,FHA insured,Cash out refinance,Male,626,136,0.217252,4.500,6.144738,7.250,True,True


Based on the segment table imported into notebook, we set the type of test.

In [5]:
testing = segs.columns.tolist()[2]
testing

'applicant_sex'

In [6]:
ref_groups

{'applicant_race_1': 'White',
 'applicant_sex': 'Male',
 'applicant_age_above_62': 'No'}

In [7]:
segs.loan_type.unique().tolist()

['Conventional', 'FHA insured']

These are dropped since they shouldn't make it into variable selection.

In [8]:
drop_this = [
    'applicant_ethnicity_1',        
    'applicant_race_1',                  
    'applicant_sex', 
    'applicant_age_above_62'
]

# Functions

Define some functions used in estimating models. These save results from each of the segmented regressions.

In [9]:
def save_summary(
    summary_out, # summary table
    res, # model summary object
    step, # uw or price
    model, # 0 or 1
    # j, # outerseg
    # i, # inner seg
    # iii # test
):

    import pandas as pd
    
    # making tmp summary table
    tmp = pd.DataFrame(columns = [
        'loan type',
        'loan purpose', # loan purp
        'step', # price or uw
        'model', # 0 or 1
        'protected basis',  # female 
        'metric', # pval, coef, r2 **
        'value' #**
    ])
    
    
    
    # save coef
    tmp2 = pd.DataFrame({
        "metric": 'param',
        "value": [res.params.values[1]],
    })

    tmp = pd.concat([tmp, tmp2], axis=0, ignore_index=True)
    
    # save p value
    tmp2 = pd.DataFrame({
        "metric": 'pval',
        "value": [res.pvalues.values[1]],
    })
    
    tmp = pd.concat([tmp, tmp2], axis=0, ignore_index=True)
    
    
    # save model fit
    
    if step == 'price':
    
    
        # ols
        tmp2 = pd.DataFrame({
            "metric": 'adj R2',
            "value": [res.rsquared_adj],
        })

        tmp = pd.concat([tmp, tmp2], axis=0, ignore_index=True)
        

    elif step == 'uw':

        # logit
        tmp2 = pd.DataFrame({
            "metric": 'psu R2',
            "value": [1 - (res.llf / res.llnull)],
        })

        tmp = pd.concat([tmp, tmp2], axis=0, ignore_index=True)


        
    # save other groups for pivots
    tmp['loan type'] = type_i
    tmp['loan purpose'] = purp_i
    tmp['protected basis'] = group_i
    tmp['step'] = step
    tmp['model'] = model
    
    
    summary_out = pd.concat([summary_out, tmp], axis=0, ignore_index=True)
    
    
    return summary_out

# Regression Analysis

This is the core of the code running regressions for both underwriting and pricing. The are 


- filter data set by segments and protected basis and reference groups
- estimate regression models for underwriting and pricing
    - Model 0: marginal effect of PB
    - Model 1: effect of PB controlling for consumer and loan attributes
    - Model 2: unbiased effect of PB controlling for consumer and loan attributes
        - estimate propensity score as the probability of PB
        - match treated (PB) observations to control observations (reference) using nearest neighbor on their estimated propensity scores
        - use matched pair data set in regression

For multivariate regression, the following approach to selection variables is used.

- rank all variables by univariate significance
- calculate correlation and apply correlation criteria in order of rank
- use forward selection with out of sample performance

The approach is similar to using varclust in SAS (svd's) to determine factors (groups) of variables that are correlated with their own group and uncorrelated with other groups. In the end keeping only most predictive variables from each group that will not have high correlation with other variables in final model.

In [10]:
%%capture output

reg_summary = pd.DataFrame(columns = [
    'loan type',
    'loan purpose', # loan purp
    'step', # price or uw
    'model', # 0 or 1 or 2
    'protected basis',  # female 
    'metric', # pval, coef, r2 **
    'value' #**
])

pred_summary = pd.DataFrame(columns = [
    'loan type',
    'loan purpose', # loan purp
    'step', # price or uw
    'model', # 0 or 1 or 2
    'protected basis',  # female 
    'actual', #
    'prediction' # 
])



for type_i in segs.loan_type.unique().tolist():
    print(f'\nType: {type_i}')
    
    for purp_i in segs[segs['loan_type'] == type_i].loan_purpose.unique().tolist():
        print(f'\n Purpose: {purp_i}')

        group_list = segs[(segs['loan_type'] == type_i)&(segs['loan_purpose'] == purp_i)][testing].unique().tolist()
        group_list.remove(ref_groups[testing])
        for group_i in group_list:
            print(f'\n  PB: {group_i}')
            print(f'  Ref: {ref_groups[testing]}')
    
            """
            Underwriting
            """

            print('\nUnderwriting')
    
            df_tmp = df[
                (df.loan_type == type_i) &
                (df.loan_purpose == purp_i) &
                (df[testing].isin([ref_groups[testing]] +  [group_i])) &
                (df['reg_uw'] == 1)
            ]



            
            
            print(f'shape: {df_tmp.shape}')

            print(df_tmp.groupby(testing).agg(
                count=('denied','size'),
                event_count=('denied','sum')
            ))

            # make dummy for protected basis group

            df_tmp[group_i] = (df_tmp[testing] == group_i).astype(int)

            df_tmp.drop(columns = drop_this, inplace = True)
            
            # drop columns with single value. will end up being things like indicator for segment
            df_tmp = df_tmp.loc[:, df_tmp.nunique(dropna=False) > 1]
            

            print('')
            print(df_tmp[group_i].value_counts())
            
            """
            UW Model 0
            """

            print('\nModel 0')
            
            import statsmodels.api as sm
            
            X = sm.add_constant(df_tmp[group_i])
            y = df_tmp["denied"]
            
            
            model = sm.Logit(y, X)
            result = model.fit()
            
            print(result.summary())
    
            reg_summary = save_summary(reg_summary, result, 'uw', 'Model 0')

                      
            # pred_tmp = pd.DataFrame({
            #     "actual": y.to_numpy(),
            #     "pred": np.asarray(result.predict(X))
            # })


            pred_tmp = pd.DataFrame({
                "actual": y,
                "prediction": result.predict(X)
            }, index=df_tmp.index)
                        
            
            pred_tmp['loan type'] = type_i
            pred_tmp['loan purpose'] = purp_i
            pred_tmp['step'] = 'uw'
            pred_tmp['model'] = 'Model 0'
            pred_tmp['protected basis'] = group_i

            pred_summary = pd.concat([pred_summary, pred_tmp], axis=0, ignore_index=True)


            
            
            """
            UW Model 1
            """
            print('\nModel 1')

            result, model_data = mod.logistic_woe_run(df_tmp,group_i)

            reg_summary = save_summary(reg_summary, result, 'uw', 'Model 1')


            # In-sample predicted probabilities
            pred_tmp = pd.DataFrame({
                "actual": y,
                "prediction": result.predict(model_data)
            }, index=df_tmp.index)
                        
            
            pred_tmp['loan type'] = type_i
            pred_tmp['loan purpose'] = purp_i
            pred_tmp['step'] = 'uw'
            pred_tmp['model'] = 'Model 1'
            pred_tmp['protected basis'] = group_i

            pred_summary = pd.concat([pred_summary, pred_tmp], axis=0, ignore_index=True)

            
            
            
            """
            UW Model 2
            """
            print('\nModel 2')

            df_tmp_psa = mod.make_match_pair(df_tmp,'denied',group_i)
            
            result, model_data  = mod.logistic_woe_run(df_tmp_psa,group_i)

            print({f'data out from function: {model_data.shape}'})
            print(model_data.head())

            reg_summary = save_summary(reg_summary, result, 'uw', 'Model 2')


            # In-sample predicted probabilities
            # this fills in preds as nans
            # pred_tmp = pd.DataFrame({
            #     "actual": y,
            #     "prediction": result.predict(model_data)
            # }, index=df_tmp.index)

            # test works
            pred_tmp = pd.DataFrame({
                "actual": np.asarray(result.model.endog).ravel(),
                "prediction": np.asarray(result.predict(model_data)).ravel()
            })

            print(pred_tmp.head())
            
            
            pred_tmp['loan type'] = type_i
            pred_tmp['loan purpose'] = purp_i
            pred_tmp['step'] = 'uw'
            pred_tmp['model'] = 'Model 2'
            pred_tmp['protected basis'] = group_i

            print(pred_tmp.head())
            
            pred_summary = pd.concat([pred_summary, pred_tmp], axis=0, ignore_index=True)



            
            
    
            """
            Pricing
            """
            print('\nPricing')
                
            df_tmp = df[
                (df.loan_type == type_i) &
                (df.loan_purpose == purp_i) &
                (df[testing].isin([ref_groups[testing]] +  [group_i])) &
                (df['reg_price'] == 1)
            ]



            
            
            print(f'shape: {df_tmp.shape}')

            # make dummy for protected basis group

            df_tmp[group_i] = (df_tmp[testing] == group_i).astype(int)

            df_tmp.drop(columns = drop_this, inplace = True)
            
            # drop columns with single value. will end up being things like indicator for segment
            df_tmp = df_tmp.loc[:, df_tmp.nunique(dropna=False) > 1]
            

            print('')
            print(df_tmp[group_i].value_counts())
    
            
            """
            Pri Model 0
            """
            print('\nModel 0')
            
            import statsmodels.api as sm
            
            X = sm.add_constant(df_tmp[group_i])
            y = df_tmp["interest_rate"]
            
            
            model = sm.OLS(y, X)
            result = model.fit()
            
            print(result.summary())
            
            reg_summary = save_summary(reg_summary, result, 'price', 'Model 0')
    


            # In-sample predicted probabilities
            pred_tmp = pd.DataFrame({
                "actual": y,
                "prediction": result.predict(X)
            }, index=df_tmp.index)
                        
            
            pred_tmp['loan type'] = type_i
            pred_tmp['loan purpose'] = purp_i
            pred_tmp['step'] = 'price'
            pred_tmp['model'] = 'Model 0'
            pred_tmp['protected basis'] = group_i

            pred_summary = pd.concat([pred_summary, pred_tmp], axis=0, ignore_index=True)




        
    
            
            """
            Pri Model 1
            """
            print('\nModel 1')

            result, model_data = mod.ols_dummy_run(df_tmp,group_i)

            print(result.summary())
            
            reg_summary = save_summary(reg_summary, result, 'price', 'Model 1')


            # In-sample predicted probabilities
            pred_tmp = pd.DataFrame({
                "actual": y,
                "prediction": result.predict(model_data)
            }, index=df_tmp.index)
                        
            
            pred_tmp['loan type'] = type_i
            pred_tmp['loan purpose'] = purp_i
            pred_tmp['step'] = 'price'
            pred_tmp['model'] = 'Model 1'
            pred_tmp['protected basis'] = group_i

            pred_summary = pd.concat([pred_summary, pred_tmp], axis=0, ignore_index=True)


            
    
            """
            Pri Model 2
            """

            print('\nModel 2')

            #pair function needs to ignore something, gets excluded later bc singular value
            df_tmp['rnd'] = 999
            df_tmp['action_taken'] = 'junk'
            
            df_tmp_psa = mod.make_match_pair(df_tmp,'rnd',group_i)
            result, model_data = mod.ols_dummy_run(df_tmp_psa,group_i)

            print(result.summary())
            
            reg_summary = save_summary(reg_summary, result, 'price', 'Model 2')


             # In-sample predicted probabilities
            # pred_tmp = pd.DataFrame({
            #     "actual": y,
            #     "prediction": result.predict(model_data)
            # }, index=df_tmp.index)

            pred_tmp = pd.DataFrame({
                "actual": np.asarray(result.model.endog).ravel(),
                "prediction": np.asarray(result.predict(model_data)).ravel()
            })
                        
            
            pred_tmp['loan type'] = type_i
            pred_tmp['loan purpose'] = purp_i
            pred_tmp['step'] = 'price'
            pred_tmp['model'] = 'Model 2'
            pred_tmp['protected basis'] = group_i

            pred_summary = pd.concat([pred_summary, pred_tmp], axis=0, ignore_index=True)

            
            
    #         break # group_i

    #     break # purp_i
    # break # type_i




        

Code output is saved to file rather than notebook since it will not display notebook in Github with a long log outupt.

In [11]:
with open(f"../data/regression analysis output {testing}.txt", "w") as f:
    f.write(output.stdout)
    f.write(output.stderr)


# Save Regression Results

Results saved contain the segment, underwriting or pricing, model, PB test, and metric. Metrics evaluated are coefficient estimate for effect of PB on target (denial and approval), p value for significance, and model fit (pseudo r2 for logistic and adjusted r2 for OLS)

In [12]:
reg_summary.head()

,loan type,loan purpose,step,model,protected basis,metric,value
0,Conventional,Cash out refinance,uw,Model 0,Female,param,-0.255703
1,Conventional,Cash out refinance,uw,Model 0,Female,pval,0.000006
2,Conventional,Cash out refinance,uw,Model 0,Female,psu R2,0.002553
3,Conventional,Cash out refinance,uw,Model 1,Female,param,-0.266475
4,Conventional,Cash out refinance,uw,Model 1,Female,pval,0.000211


In [13]:
reg_summary.to_pickle(f"../data/04_df_reg_{testing}.pkl")
# reg_summary = pd.read_pickle("./data/reg_summary.pkl")

# Save In sample Predictions

In sample predictions are also saved. These are used to show the difference in predictions such as separation in logistic.

In [14]:
pred_summary.head()

,loan type,loan purpose,step,model,protected basis,actual,prediction
0,Conventional,Cash out refinance,uw,Model 0,Female,0,0.221796
1,Conventional,Cash out refinance,uw,Model 0,Female,0,0.1808
2,Conventional,Cash out refinance,uw,Model 0,Female,0,0.221796
3,Conventional,Cash out refinance,uw,Model 0,Female,0,0.221796
4,Conventional,Cash out refinance,uw,Model 0,Female,0,0.221796


In [15]:
pred_summary.to_pickle(f"../data/04_df_pred_{testing}.pkl")